In [ ]:
//IMPORTS

import cv2
import numpy as np
from pynq import Overlay, allocate
from pynq.lib import DMA


In [ ]:
//USER CONFIGURATION

# =========================
# VIVADO-ASSIGNED ADDRESSES
# =========================

DMA_NAME = "axi_dma_0"      # EXACT IP name in block design
CTRL_NAME = "top_axis_IP_0"    # Your AXI-Lite control IP name

# =========================
# FILE PATHS (SD CARD)
# =========================

BIT_PATH     = "/home/xilinx/overlays/snn.bit"
WEIGHT_DIR   = "/home/xilinx/weights"
INPUT_VIDEO  = "/home/xilinx/input.mp4"
OUTPUT_VIDEO = "/home/xilinx/output_2x.mp4"

# =========================
# DATA CONSTANTS
# =========================

H = 224
W = 224
CHANNELS = 6
INPUT_SIZE = CHANNELS * H * W   # 301056
FLOW_SIZE  = 2 * H * W

Q15_SCALE = 32768
FLOW_SCALE = 1.0 / 32768.0
INTERP_T = 0.5


In [ ]:
//LOAD BITSTREAM

overlay = Overlay(BIT_PATH)
overlay.download()

print("✅ Bitstream loaded")


In [ ]:
//GET DMA AND CONTROL HANDLES

dma = overlay.axi_dma_0        # MUST match DMA_NAME
ctrl = overlay.snn_ctrl_0      # MUST match CTRL_NAME

print("✅ DMA and control IP connected")


In [ ]:
//ALLOCATE DDR BUFFERS

dma_in  = allocate(shape=(INPUT_SIZE,), dtype=np.int16)
dma_out = allocate(shape=(FLOW_SIZE,), dtype=np.int16)

print("✅ DDR buffers allocated")


In [ ]:
//LOAD WEIGHTS INTO FPGA

def load_weights(ctrl, weight_path, base_reg):
    data = np.fromfile(weight_path, dtype=np.int16)
    for i, v in enumerate(data):
        ctrl.write(base_reg + i*4, int(v) & 0xFFFF)


In [ ]:
//LOAD LIF STATS

lif = np.fromfile(f"{WEIGHT_DIR}/lif1_stats.bin", dtype=np.float32)
beta, threshold = lif

ctrl.write(0x20, int(beta * (1<<16)))
ctrl.write(0x24, int(threshold * (1<<16)))

print("✅ LIF parameters loaded")


In [ ]:
//PRE - PROCESSING CODE

class VideoPreprocessorQ15:
    def __init__(self, video_path):
        self.cap = cv2.VideoCapture(video_path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot open video: {video_path}")

        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        if self.fps <= 0:
            self.fps = 30.0

        self.width  = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

        self.prev_frame = self._read_frame()

    def _read_frame(self):
        ret, frame = self.cap.read()
        if not ret:
            return None
        return frame

    def preprocess_pair(self, frame0, frame1):
        f0 = cv2.resize(frame0, (224, 224))
        f1 = cv2.resize(frame1, (224, 224))

        f0 = f0.astype(np.float32) / 255.0
        f1 = f1.astype(np.float32) / 255.0

        f0 = 2.0 * f0 - 1.0
        f1 = 2.0 * f1 - 1.0

        f0 = np.clip(f0 * 32768, -32768, 32767).astype(np.int16)
        f1 = np.clip(f1 * 32768, -32768, 32767).astype(np.int16)

        f0 = f0.transpose(2, 0, 1)
        f1 = f1.transpose(2, 0, 1)

        return np.concatenate([f0, f1], axis=0)

    def get_next_tensor(self):
        if self.prev_frame is None:
            return None, None, None

        curr_frame = self._read_frame()
        if curr_frame is None:
            return None, None, None

        input_tensor = self.preprocess_pair(self.prev_frame, curr_frame)

        frame0 = self.prev_frame
        frame1 = curr_frame
        self.prev_frame = curr_frame

        return input_tensor, frame0, frame1

    def release(self):
        self.cap.release()


In [ ]:
//FUNCTION OF PRE - PROCESSING

preprocessor = VideoPreprocessorQ15(INPUT_VIDEO)


In [ ]:
//POST - PROCESSING CODE

# ------------------------------------------------------------
# WARP FUNCTION
# ------------------------------------------------------------
def warp_frame(frame, flow, direction):
    h, w = frame.shape[:2]

    dx = flow[0].astype(np.float32) * FLOW_SCALE * direction
    dy = flow[1].astype(np.float32) * FLOW_SCALE * direction

    x, y = np.meshgrid(
        np.arange(w, dtype=np.float32),
        np.arange(h, dtype=np.float32)
    )

    map_x = x + dx
    map_y = y + dy

    warped = cv2.remap(
        frame,
        map_x,
        map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT
    )

    return warped


# ------------------------------------------------------------
# POST-PROCESS + VIDEO WRITE
# ------------------------------------------------------------
def postprocess_and_write(
    frame0_full,
    frame1_full,
    flow,
    video_writer,
    resize_h=224,
    resize_w=224
):
    # 1️⃣ Write original frame
    video_writer.write(frame0_full)

    # 2️⃣ Resize frames to flow resolution
    f0 = cv2.resize(frame0_full, (resize_w, resize_h))
    f1 = cv2.resize(frame1_full, (resize_w, resize_h))

    # 3️⃣ Warp
    w0 = warp_frame(f0, flow, +INTERP_T)
    w1 = warp_frame(f1, flow, -(1.0 - INTERP_T))

    # 4️⃣ Blend
    interp_small = (
        (1.0 - INTERP_T) * w0.astype(np.float32) +
        INTERP_T * w1.astype(np.float32)
    )

    interp_small = np.clip(interp_small, 0, 255).astype(np.uint8)

    # 5️⃣ Upscale to original resolution
    h0, w0 = frame0_full.shape[:2]
    interp_full = cv2.resize(
        interp_small,
        (w0, h0),
        interpolation=cv2.INTER_LINEAR
    )

    # 6️⃣ Write interpolated frame
    video_writer.write(interp_full)



In [ ]:
//VIDEO WRITER SETUP

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps*2, (width, height))

print("✅ Video writer ready")


In [ ]:
//MAIN EXECUTION LOOP

while True:
    input_tensor, frame0, frame1 = preprocessor.get_next_tensor()
    if input_tensor is None:
        break

    # -------------------------
    # COPY PREPROCESS → DDR
    # -------------------------
    dma_in[:] = input_tensor.flatten()

    # -------------------------
    # START DMA
    # -------------------------
    dma.recvchannel.transfer(dma_out)
    dma.sendchannel.transfer(dma_in)

    dma.sendchannel.wait()
    dma.recvchannel.wait()

    # -------------------------
    # RESHAPE FLOW
    # -------------------------
    flow = dma_out.reshape(2, H, W)

    # -------------------------
    # POSTPROCESS
    # -------------------------
    postprocess_and_write(
        frame0,
        frame1,
        flow,
        writer,
        resize_h=H,
        resize_w=W
    )


In [ ]:
//CLEAN UP

preprocessor.release()
writer.release()

dma_in.freebuffer()
dma_out.freebuffer()

print("🎉 DONE — Video saved to:", OUTPUT_VIDEO)
